In [1]:
"""
represent_E_on_V.py
====================
Attempt to construct an explicit matrix representation
of the Fano-TRB algebra E (9-dim) on a 144-dim space V.

Approaches:
  A. V = E ⊗ R^16 with ρ(x) = L_x ⊗ I_16
  B. V = O ⊗ R^18 with ρ(e_i) = L_i^(oct) ⊗ I_18
  C. V = R^24 ⊗ R^6 with L_i^(oct) ⊗ I_3 acting on R^24

For each, find the invariant subspaces and check for 137.

Note: E is non-associative, so "representation" means a
linear map ρ: E → End(V) preserving the relations:
  ρ(e_i)² = -I_V
  ρ(t)² = 0
  ρ(t)ρ(e_i) = 3ρ(e_i)
  ρ(e_i)ρ(t) = -3ρ(e_i)
"""

import numpy as np
from itertools import product
from scipy.linalg import null_space

# =============================================================
# OCTONION LEFT-MULTIPLICATION MATRICES (8×8)
# =============================================================
# Octonion basis: 1 (index 0), e_1, e_2, ..., e_7 (indices 1-7)
# But for consistency with Fano indexing (e_0..e_6 for the 7
# imaginary units), we'll use 1 + e_0..e_6.

# Fano lines (indices 0-6):
FANO_LINES = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]

# Build octonion multiplication table
# Indices: 0 = identity, 1-7 = imaginary units e_1..e_7
# But we'll rename: oct[0] = 1, oct[i+1] = e_i for i=0..6

N_OCT = 8
oct_mult = np.zeros((N_OCT, N_OCT, N_OCT))

# Identity
for i in range(N_OCT):
    oct_mult[0, i, i] = 1
    oct_mult[i, 0, i] = 1

# Fano-based imaginary multiplication (with our 0-6 labeling)
for (a, b, c) in FANO_LINES:
    # In octonion basis: e_a, e_b, e_c at indices a+1, b+1, c+1
    ia, ib, ic = a+1, b+1, c+1
    oct_mult[ia, ib, ic] = 1
    oct_mult[ib, ic, ia] = 1
    oct_mult[ic, ia, ib] = 1
    oct_mult[ib, ia, ic] = -1
    oct_mult[ic, ib, ia] = -1
    oct_mult[ia, ic, ib] = -1

# e_i² = -1
for i in range(1, 8):
    oct_mult[i, i, 0] = -1


def left_mult_matrix(idx):
    """8×8 matrix for left multiplication by octonion basis element idx."""
    L = np.zeros((N_OCT, N_OCT))
    for j in range(N_OCT):
        for k in range(N_OCT):
            L[k, j] = oct_mult[idx, j, k]
    return L


# Build L_i for imaginary units e_0..e_6 (oct indices 1..7)
L_oct = [left_mult_matrix(i+1) for i in range(7)]

# Verify: L_i² = -I
print("=" * 70)
print("VERIFY OCTONION LEFT-MULTIPLICATION")
print("=" * 70)
for i in range(7):
    check = np.allclose(L_oct[i] @ L_oct[i], -np.eye(N_OCT))
    print(f"  L_{i}² = -I? {check}")
print()

# =============================================================
# APPROACH A: V = E ⊗ R^16  (9 × 16 = 144)
# =============================================================
print("=" * 70)
print("APPROACH A: V = E ⊗ R^16")
print("=" * 70)
print()

# Build left-multiplication matrices on E (9-dim)
# Basis: e_0..e_6 (indices 0-6), 1 (index 7), t (index 8)

N_E = 9
MOD = 9

# Build the algebra E's multiplication table over R (not mod 9)
E_mult = np.zeros((N_E, N_E, N_E))

# Identity at 7
for i in range(N_E):
    E_mult[7, i, i] = 1
    E_mult[i, 7, i] = 1

# Fano rules
for (a, b, c) in FANO_LINES:
    E_mult[a, b, c] = 1
    E_mult[b, c, a] = 1
    E_mult[c, a, b] = 1
    E_mult[b, a, c] = -1
    E_mult[c, b, a] = -1
    E_mult[a, c, b] = -1

# e_i² = -1 (index 7 is identity)
for i in range(7):
    E_mult[i, i, 7] = -1

# TRB rules (candidate E):
# t² = 0
# t·e_i = 3e_i  (so E_mult[8, i, i] = 3)
# e_i·t = -3e_i (so E_mult[i, 8, i] = -3)
# t·1 = 3 (so E_mult[8, 7, 7] = 3)
# 1·t = t (so E_mult[7, 8, 8] = 1)

for i in range(7):
    E_mult[8, i, i] = 3
    E_mult[i, 8, i] = -3

E_mult[8, 7, 7] = 3    # t·1 = 3
E_mult[7, 8, 8] = 1    # 1·t = t
# t·t = 0 (already 0)


def left_mult_E(idx):
    """9×9 left multiplication matrix on E."""
    L = np.zeros((N_E, N_E))
    for j in range(N_E):
        for k in range(N_E):
            L[k, j] = E_mult[idx, j, k]
    return L


L_E = [left_mult_E(i) for i in range(7)]    # e_0..e_6
L_t = left_mult_E(8)                        # t
L_1 = left_mult_E(7)                        # identity

# Verify relations
print("Verify relations on E:")
for i in range(7):
    check = np.allclose(L_E[i] @ L_E[i], -np.eye(N_E))
    print(f"  L_{i}² = -I? {check}")
print(f"  L_t² = 0? {np.allclose(L_t @ L_t, 0)}")
print(f"  L_t L_0 = 3 L_0? {np.allclose(L_t @ L_E[0], 3 * L_E[0])}")
print(f"  L_0 L_t = -3 L_0? {np.allclose(L_E[0] @ L_t, -3 * L_E[0])}")
print()

# Now tensor with I_16 to get 144-dim
I_16 = np.eye(16)
rho_e = [np.kron(L, I_16) for L in L_E]
rho_t = np.kron(L_t, I_16)

print(f"ρ(e_i) shape: {rho_e[0].shape}")
print(f"ρ(t) shape:   {rho_t.shape}")
print()

# Verify on 144-dim
print("Verify relations on V_144:")
print(f"  ρ(e_0)² = -I? {np.allclose(rho_e[0] @ rho_e[0], -np.eye(144))}")
print(f"  ρ(t)² = 0? {np.allclose(rho_t @ rho_t, 0)}")
print(f"  ρ(t)ρ(e_0) = 3ρ(e_0)? {np.allclose(rho_t @ rho_e[0], 3 * rho_e[0])}")
print()


# Find invariant subspaces via simultaneous eigenspaces
# An invariant subspace W is one preserved by all ρ(e_i) and ρ(t).
# The simplest invariant subspaces are the joint eigenspaces.

# Actually, let's use a different approach: build the algebra
# generated by {rho_e[i], rho_t} and find its commutant.

# But that's expensive. Let's use a simpler approach:
# Compute the common null space of all (ρ(x) - λI) for eigenvalues λ.

# Simpler: compute the joint invariant subspaces by computing
# the "generalized weight spaces".

# Actually, the cleanest test: compute the commutant algebra
# (operators commuting with all ρ(x)) and check its dimension.

# For our tensored construction, the commutant is M_16(R).
# So the invariant subspaces are all W_1 ⊗ W_2 with W_1 invariant
# under the E-action and W_2 ⊆ R^16 arbitrary.

# Invariant subspaces of E: F (= span(e_0..e_6), 7-dim) and E (9-dim).
# Any smaller invariant subspace?

# Test: is span(e_0..e_6) invariant under all ρ(e_i)?
F = np.zeros((N_E, N_E))
for i in range(7):
    F[i, i] = 1  # project onto e_0..e_6

invariant_F = all(
    np.allclose(L @ F, F @ (L @ F))
    for L in L_E
)
print(f"Is F = span(e_0..e_6) invariant under L_{{'e_i'}}? {invariant_F}")

# Also check: is F invariant under L_t?
invariant_F_t = np.allclose(L_t @ F, F @ (L_t @ F))
print(f"Is F invariant under L_t? {invariant_F_t}")
print()

# So F is a 7-dim invariant subspace of E.
# On V_144, this lifts to F ⊗ R^16, dimension 7 × 16 = 112.
print(f"Invariant subspace F ⊗ R^16 has dimension: 7 × 16 = 112")
print(f"Target: 137")
print(f"Match: {112 == 137}")
print()

# =============================================================
# APPROACH B: V = O ⊗ R^18  (8 × 18 = 144)
# =============================================================
print("=" * 70)
print("APPROACH B: V = O ⊗ R^18")
print("=" * 70)
print()

I_18 = np.eye(18)
rho_e_B = [np.kron(L, I_18) for L in L_oct]

print(f"ρ(e_i) shape: {rho_e_B[0].shape}")
print()

# Invariant subspaces of O under L_i?
# O is irreducible under the left-multiplication action
# (the only invariant subspaces are 0 and O itself).

# Check: is {0} the only proper invariant subspace?
# Since O is 8-dim and the action is known to be irreducible,
# invariant subspaces of V_144 are of the form O ⊗ W for W ⊆ R^18.
# Dimensions: 8k for k = 0, ..., 18.
print("Invariant subspaces of O ⊗ R^18:")
print("  Form: O ⊗ W for any W ⊆ R^18")
print("  Dimensions: 8k for k = 0, 1, ..., 18")
print("  Possible dims: 0, 8, 16, 24, 32, ..., 144")
print(f"  137 is not in this set. Match: False")
print()

# =============================================================
# APPROACH C: V = R^24 ⊗ R^6 with L_i ⊗ I_3 on R^24
# =============================================================
print("=" * 70)
print("APPROACH C: V = R^24 ⊗ R^6")
print("=" * 70)
print()

# R^24 = O ⊗ R^3 (since O is 8-dim)
I_3 = np.eye(3)
L_i_24 = [np.kron(L, I_3) for L in L_oct]

I_6 = np.eye(6)
rho_e_C = [np.kron(L_24, I_6) for L_24 in L_i_24]

print(f"ρ(e_i) shape: {rho_e_C[0].shape}")
print()

# Invariant subspaces: same as approach B, 8k dims.
print("Invariant subspaces: 8k for k = 0, ..., 18 (as in B)")
print(f"  137 is not in this set. Match: False")
print()

# =============================================================
# APPROACH D: V = R^12 ⊗ R^12, look for natural action
# =============================================================
print("=" * 70)
print("APPROACH D: V = R^12 ⊗ R^12")
print("=" * 70)
print()

# 12 = number of icosahedral vertices (our previous derivation)
# But how does E act on R^12? No obvious action.
# Skip this approach for now.
print("No obvious action of E on R^12. Skipping.")
print()

# =============================================================
# SUMMARY
# =============================================================
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print()
print("Approaches tried:")
print("  A. V = E ⊗ R^16:")
print("     Invariant subspaces: 0, 112, 128, 144.")
print("     137 not present.")
print()
print("  B. V = O ⊗ R^18:")
print("     Invariant subspaces: dims 8k for k=0..18.")
print("     137 not present.")
print()
print("  C. V = R^24 ⊗ R^6:")
print("     Same as B. 137 not present.")
print()
print("  D. V = R^12 ⊗ R^12:")
print("     No natural action.")
print()
print("=" * 70)
print("CONCLUSION")
print("=" * 70)
print()
print("None of the natural constructions produces a 137-dim")
print("invariant subspace. The 137 does NOT appear from the")
print("algebra E acting on any of these spaces.")
print()
print("Possible interpretations:")
print("  1. E does not act naturally on V_144 (no rep exists).")
print("  2. The action exists but the invariant subspace is not")
print("     natural; it requires additional structure.")
print("  3. The derivation of N = 137 is PHYSICAL (counting),")
print("     not algebraic (representation-theoretic).")
print()
print("Honest status: The physical derivation of N = 137 stands.")
print("A purely algebraic derivation from E remains open.")

VERIFY OCTONION LEFT-MULTIPLICATION
  L_0² = -I? True
  L_1² = -I? True
  L_2² = -I? True
  L_3² = -I? True
  L_4² = -I? True
  L_5² = -I? True
  L_6² = -I? True

APPROACH A: V = E ⊗ R^16

Verify relations on E:
  L_0² = -I? False
  L_1² = -I? False
  L_2² = -I? False
  L_3² = -I? False
  L_4² = -I? False
  L_5² = -I? False
  L_6² = -I? False
  L_t² = 0? False
  L_t L_0 = 3 L_0? False
  L_0 L_t = -3 L_0? False

ρ(e_i) shape: (144, 144)
ρ(t) shape:   (144, 144)

Verify relations on V_144:
  ρ(e_0)² = -I? False
  ρ(t)² = 0? False
  ρ(t)ρ(e_0) = 3ρ(e_0)? False

Is F = span(e_0..e_6) invariant under L_{'e_i'}? False
Is F invariant under L_t? True

Invariant subspace F ⊗ R^16 has dimension: 7 × 16 = 112
Target: 137
Match: False

APPROACH B: V = O ⊗ R^18

ρ(e_i) shape: (144, 144)

Invariant subspaces of O ⊗ R^18:
  Form: O ⊗ W for any W ⊆ R^18
  Dimensions: 8k for k = 0, 1, ..., 18
  Possible dims: 0, 8, 16, 24, 32, ..., 144
  137 is not in this set. Match: False

APPROACH C: V = R^24 ⊗ R^6
